In [ ]:
Bước 1: Kết nối với Google Drive và khai báo thư viện
Bước này giúp cấp quyền cho Colab lưu file mô hình AI vào Google Drive của bạn.

In [1]:
# Ô code 1: Kết nối Drive và nạp thư viện
from google.colab import drive
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
import os

# Yêu cầu quyền truy cập Google Drive
drive.mount('/content/drive')

print("Phiên bản TensorFlow:", tf.__version__)

Mounted at /content/drive
Phiên bản TensorFlow: 2.21.0


Bước 2: Tải dữ liệu nông nghiệp từ Internet
Để demo, chúng ta dùng tập dữ liệu phân loại 5 loại thực vật có sẵn của Google. Dữ liệu sẽ được tự động tải về và chia thành tập huấn luyện (training) và tập kiểm tra (validation).

In [2]:
# Ô code 2: Tải và chuẩn bị dữ liệu
dataset_url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
data_dir = tf.keras.utils.get_file('flower_photos', origin=dataset_url, untar=True)

batch_size = 32
img_height = 160
img_width = 160

# Tạo tập dữ liệu để AI học (80% dữ liệu)
train_ds = tf.keras.utils.image_dataset_from_directory(
  data_dir, validation_split=0.2, subset="training", seed=123,
  image_size=(img_height, img_width), batch_size=batch_size)

# Tạo tập dữ liệu để kiểm tra AI (20% dữ liệu)
val_ds = tf.keras.utils.image_dataset_from_directory(
  data_dir, validation_split=0.2, subset="validation", seed=123,
  image_size=(img_height, img_width), batch_size=batch_size)

class_names = train_ds.class_names
print("Các nhãn thực vật AI sẽ học:", class_names)

228813984/228813984 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
Found 3670 files belonging to 1 classes.
Using 2936 files for training.
Found 3670 files belonging to 1 classes.
Using 734 files for validation.
Các nhãn thực vật AI sẽ học: ['flower_photos']


Bước 3: Xây dựng và Huấn luyện mô hình (AI Learning)
Chúng ta sẽ sử dụng một kỹ thuật gọi là Transfer Learning (Học chuyển giao) với mô hình MobileNetV2. Cách này giúp AI học cực kỳ nhanh, độ chính xác cao mà không cần máy tính quá mạnh.

In [ ]:
# Ô code 3: Xây dựng và huấn luyện AI
# Sử dụng bộ não AI đã được đào tạo sẵn (MobileNetV2)
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(img_height, img_width, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False # Giữ nguyên kiến thức nền

# Gắn thêm các lớp mới để học nhận diện tập dữ liệu thực vật của chúng ta
model = tf.keras.Sequential([
  tf.keras.layers.Rescaling(1./127.5, offset=-1), # Chuẩn hóa ảnh
  base_model,
  tf.keras.layers.GlobalAveragePooling2D(),
  tf.keras.layers.Dense(len(class_names), activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

print("Bắt đầu cho AI học (Huấn luyện)...")
# Cho AI học trong 3 vòng (epochs). Có thể tăng lên 10-20 vòng nếu muốn AI thông minh hơn.
history = model.fit(train_ds, validation_data=val_ds, epochs=3)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Bắt đầu cho AI học (Huấn luyện)...
Epoch 1/3


/usr/local/lib/python3.13/dist-packages/keras/src/ops/nn.py:947: UserWarning: You are using a softmax over axis -1 of a tensor of shape (None, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(


92/92 ━━━━━━━━━━━━━━━━━━━━ 102s 1s/step - accuracy: 0.0000e+00 - loss: 0.0000e+00 - val_accuracy: 0.0000e+00 - val_loss: 0.0000e+00
Epoch 2/3
28/92 ━━━━━━━━━━━━━━━━━━━━ 47s 739ms/step - accuracy: 0.0000e+00 - loss: 0.0000e+00

Bước 4: Lưu bộ não AI vào Google Drive
Lưu trữ mô hình dưới dạng file .h5. Bạn có thể tải file này về máy tính cá nhân ở nhà, hoặc mang sang bất kỳ máy nào có cài Python để sử dụng lại mà không cần huấn luyện lại từ đầu.

In [ ]:
# Ô code 4: Lưu mô hình
model_path = '/content/drive/MyDrive/nong_nghiep_model.h5'
model.save(model_path)
print(f"✅ Đã lưu thành công mô hình AI tại: {model_path}")

Bước 5: Trực quan hóa kết quả cho Sinh viên
Đây là phần sinh viên nông nghiệp sẽ thích nhất. Đoạn code này sẽ tải mô hình bạn vừa lưu, lấy một vài bức ảnh ngẫu nhiên, cho AI đoán và in kết quả ra màn hình với hình ảnh trực quan.

In [ ]:
# Ô code 5: Tải mô hình và nhận diện hình ảnh thực tế
# Nạp lại bộ não AI từ Drive (Giả lập việc bạn dùng mô hình ở một máy khác)
loaded_model = tf.keras.models.load_model(model_path)

# Lấy 1 lô hình ảnh để kiểm tra
plt.figure(figsize=(12, 10))
for images, labels in val_ds.take(1):
  # Cho AI dự đoán
  predictions = loaded_model.predict(images)

  # Hiển thị 9 tấm ảnh đầu tiên kèm kết quả
  for i in range(9):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(images[i].numpy().astype("uint8"))

    # Lấy kết quả AI đoán cao nhất
    predicted_class = class_names[np.argmax(predictions[i])]
    actual_class = class_names[labels[i]]

    # Đổi màu chữ: Đoán đúng màu xanh, đoán sai màu đỏ
    color = 'green' if predicted_class == actual_class else 'red'

    plt.title(f"AI đoán: {predicted_class}\nThực tế: {actual_class}", color=color, fontsize=12)
    plt.axis("off")

plt.tight_layout()
plt.show()

Cách thay bằng dữ liệu nông nghiệp thực tế của bạn:
Thay vì dùng link tải của Google ở Bước 2, bạn chỉ cần nén các hình ảnh của bạn thành 1 file .zip (bên trong chia thành các thư mục như Benh_Gia_Suong_Mai, Benh_Dom_La, La_Khoe_Manh), tải lên Google Drive, sau đó dùng lệnh !unzip /content/drive/MyDrive/du_lieu_cua_toi.zip giải nén vào Colab và truyền đường dẫn thư mục đó vào biến data_dir.